# Multi-Agent Question-Paper Generation and Validation System

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ashyx12/agentic-ai-assignment/blob/main/multi_agent_question_paper.ipynb)

**Topic selected from the assignment list:** Multi-Agent Question-Paper Generation and Validation System.

This notebook uses a small open-weight language model directly in the Colab runtime—**no OpenAI/Gemini API key**. Three agents work in sequence: **Blueprint Agent → Question Generator → Validator**. The validator decides whether a revision agent should fix the draft.

## 1. Install and load the local model
In Colab, select **Runtime → Change runtime type → T4 GPU** if available. This downloads the public model weights to the runtime; it is not a paid hosted API. You can run on CPU too, but generation may be slower.

In [ ]:
%pip -q install -U transformers accelerate
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype="auto",
    device_map="auto",
)
print(f"Loaded {MODEL_ID}; device: {model.device}")

## 2. Set the subject and syllabus
Change these values to your own course. Keep the syllabus concise for better results.

In [ ]:
subject = "Introductory Python Programming"
syllabus = "Variables and data types; conditionals; loops; functions; lists; dictionaries."
difficulty = "beginner to intermediate"
number_of_questions = 5

## 3. Agent runtime
Each agent receives a specific role and the previous agent's output. The model runs locally in this notebook.

In [ ]:
def run_agent(agent_name, instructions, context, max_new_tokens=450):
    messages = [
        {"role": "system", "content": f"You are {agent_name}. Follow instructions carefully. Be concise and factual."},
        {"role": "user", "content": f"TASK:\n{instructions}\n\nCONTEXT:\n{context}"},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.05,
            pad_token_id=tokenizer.eos_token_id,
        )
    new_tokens = output[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()


## 4. Run the agents
The workflow creates a blueprint, generates a draft, and then validates coverage, answer keys, clarity, and duplicates. If the validator requests changes, a revision agent gets one attempt to fix the draft.

In [ ]:
course_context = f"Subject: {subject}\nSyllabus: {syllabus}\nDifficulty: {difficulty}\nNumber of questions: {number_of_questions}"

# Agent 1: Blueprint Agent plans coverage and question types.
blueprint = run_agent(
    "Blueprint Agent",
    "Create a compact exam blueprint. Distribute the requested number of questions across the syllabus and difficulty level. State topic coverage and question types. Do not write the questions yet.",
    course_context,
    max_new_tokens=250,
)
print("=== BLUEPRINT AGENT ===\n", blueprint)

# Agent 2: Question Generator uses the blueprint to produce the paper and answer key.
draft = run_agent(
    "Question Generation Agent",
    "Generate the requested question paper using the blueprint. Number every question. Include a clear expected answer and a one-sentence explanation for each. Cover the syllabus and avoid duplicate questions. Use a mix of conceptual and practical questions appropriate to the stated level.",
    course_context + "\n\nBLUEPRINT:\n" + blueprint,
    max_new_tokens=800,
)
print("\n=== GENERATED DRAFT ===\n", draft)

# Agent 3: Validator evaluates the draft and selects APPROVE or REVISE.
review = run_agent(
    "Question Paper Validation Agent",
    "Review the draft against the subject, syllabus, requested question count, and difficulty. Check relevance, duplicates, ambiguity, and whether every question has a plausible answer and explanation. The FIRST line must be exactly 'DECISION: APPROVE' if there are no major issues, or 'DECISION: REVISE' if it needs fixes. Then list brief findings. Do not rewrite the paper.",
    course_context + "\n\nBLUEPRINT:\n" + blueprint + "\n\nDRAFT:\n" + draft,
    max_new_tokens=300,
)
print("\n=== VALIDATOR ===\n", review)

# Agentic routing: the validator's decision determines whether the revision agent runs.
if "DECISION: REVISE" in review.upper():
    final_paper = run_agent(
        "Question Paper Revision Agent",
        "Revise the draft using the validator's findings. Preserve correct material, repair identified problems, keep the requested number of questions, and include an answer and short explanation for each. Output only the corrected final paper.",
        course_context + "\n\nBLUEPRINT:\n" + blueprint + "\n\nDRAFT:\n" + draft + "\n\nVALIDATOR FINDINGS:\n" + review,
        max_new_tokens=850,
    )
    status = "Revised after validation"
else:
    final_paper = draft
    status = "Approved by validator (no revision requested)"

print("\n=== FINAL QUESTION PAPER ===\n")
print(final_paper)
print("\nWorkflow status:", status)

## 5. What makes the workflow agentic?
- **Planning:** the Blueprint Agent decides how to distribute syllabus coverage.
- **Execution:** the Question Generation Agent creates the draft based on that plan.
- **Evaluation and routing:** the Validator Agent checks the draft and chooses approval or revision.
- **Feedback loop:** when revision is requested, the Revision Agent uses the critique to repair the paper.

**Limitations:** this is a lightweight educational prototype. A small local model can produce incorrect answers, so review the final paper before using it. Model downloads require internet access in Colab; no paid API key is used.